# Step 0 – Setup & Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/
!rm -rf FAIMDL_PROJECT
!git clone https://github.com/raiwrd/FAIMDL_PROJECT.git
%cd /content/FAIMDL_PROJECT/eomt
!pip install -r requirements.txt


# Step 1 – Imports & Global Config

In [ ]:
import os, sys, gc, json, math, time, yaml, warnings, importlib
import numpy as np
import torch
import torch.nn.functional as F
from torch.amp.autocast_mode import autocast
from pathlib import Path
from PIL import Image
import torchvision.transforms as T
from tqdm import tqdm
from sklearn.metrics import average_precision_score, roc_curve
from lightning import seed_everything
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import RepositoryNotFoundError

seed_everything(0, verbose=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Paths ─────────────────────────────────────────────────────────────────
DRIVE_ROOT      = '/content/drive/MyDrive/FAIMDL_PROJECT'
REPO_ROOT       = '/content/FAIMDL_PROJECT/eomt'

CONFIG_CITY     = f'{REPO_ROOT}/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml'
CONFIG_COCO     = f'{REPO_ROOT}/configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml'

CKPT_CITY       = f'{DRIVE_ROOT}/checkpoints/eomt_cityscapes.bin'
CKPT_COCO       = f'{DRIVE_ROOT}/checkpoints/eomt_coco.bin'
ERFNET_WEIGHTS  = f'{REPO_ROOT}/trained_models/erfnet_pretrained.pth'

DATA_PATH       = f'{DRIVE_ROOT}/cityscapes'
SAVE_DIR        = f'{DRIVE_ROOT}/finetune_coco_to_cityscapes'
ANOMALY_ROOT    = f'{DRIVE_ROOT}/Anomaly_Validation_Datasets'
STEP7_CACHE_DIR = f'{DRIVE_ROOT}/step7_cache'
STEP7_RESULTS   = f'{DRIVE_ROOT}/results_step7.json'
STEP8_CACHE_DIR = f'{DRIVE_ROOT}/step8_cache'
STEP8_RESULTS   = f'{DRIVE_ROOT}/results_step8.json'

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(STEP7_CACHE_DIR, exist_ok=True)
os.makedirs(STEP8_CACHE_DIR, exist_ok=True)

# ── Constants ─────────────────────────────────────────────────────────────
IGNORE_LABEL = 19
CITYSCAPES_CLASS_NAMES = [
    'road', 'sidewalk', 'building', 'wall', 'fence', 'pole',
    'traffic light', 'traffic sign', 'vegetation', 'terrain', 'sky',
    'person', 'rider', 'car', 'truck', 'bus', 'train', 'motorcycle', 'bicycle',
]

sys.path.insert(0, f'{REPO_ROOT}/../eval')

warnings.filterwarnings(
    'ignore',
    message=r".*Attribute 'network' is an instance of `nn\.Module` and is already saved during checkpointing.*",
)

print(f'Setup completato. Device: {device}')


# Step 2 – Dataset & DataLoader (Cityscapes validation)

In [ ]:
with open(CONFIG_CITY, 'r') as f:
    config = yaml.safe_load(f)

dm_name, dm_cls_name = config['data']['class_path'].rsplit('.', 1)
dm_cls    = getattr(importlib.import_module(dm_name), dm_cls_name)
dm_kwargs = config['data'].get('init_args', {})

data = dm_cls(
    path=DATA_PATH, batch_size=1, num_workers=0,
    check_empty_targets=False, **dm_kwargs,
).setup()

val_loader = data.val_dataloader()
print(f'num_classes: {data.num_classes}  |  img_size: {data.img_size}')


# Step 3 – Utility Functions

Helpers used in the following steps:
- `_build_encoder / _build_network / _build_lightning_module` – model construction
- `build_model_from_config` / `build_coco_model` – loading model
- `infer_semantic` – infernce
- `build_gt_from_batch`, `remap_mask`, `evaluate_model` – IoU evaluation
- `AnomalyDataset` / `EOMTAnomalyDataset` – dataset anomaly
- `compute_metrics`, `score_msp/max_logit/max_entropy/rba` – anomaly scoring
- `DATASET_CONFIGS` – benchmarks


In [ ]:
# ════════════════════════════════════════════════════════════
# 3a – Model building helpers
# ════════════════════════════════════════════════════════════

def _build_encoder(cfg, img_size):
    enc_cfg = cfg['model']['init_args']['network']['init_args']['encoder']
    mod, cls_name = enc_cfg['class_path'].rsplit('.', 1)
    cls = getattr(importlib.import_module(mod), cls_name)
    return cls(img_size=img_size, **enc_cfg.get('init_args', {}))


def _build_network(cfg, encoder, num_classes):
    net_cfg = cfg['model']['init_args']['network']
    mod, cls_name = net_cfg['class_path'].rsplit('.', 1)
    cls = getattr(importlib.import_module(mod), cls_name)
    net_kwargs = {k: v for k, v in net_cfg['init_args'].items() if k != 'encoder'}
    return cls(masked_attn_enabled=False, num_classes=num_classes, encoder=encoder, **net_kwargs)


def _build_lightning_module(cfg, network, img_size, num_classes):
    mod, cls_name = cfg['model']['class_path'].rsplit('.', 1)
    lit_cls = getattr(importlib.import_module(mod), cls_name)
    model_kwargs = {k: v for k, v in cfg['model']['init_args'].items() if k != 'network'}
    if 'stuff_classes' in cfg['data'].get('init_args', {}):
        model_kwargs['stuff_classes'] = cfg['data']['init_args']['stuff_classes']
    return lit_cls(img_size=img_size, num_classes=num_classes, network=network, **model_kwargs)


def _resize_abs_pos_embed(pos_embed_ckpt, pos_embed_model):
    if pos_embed_ckpt.shape == pos_embed_model.shape:
        return pos_embed_ckpt
    _, n_old, c = pos_embed_ckpt.shape
    _, n_new, c2 = pos_embed_model.shape
    assert c == c2, 'Embedding dim mismatch in pos_embed'
    gs_old, gs_new = int(math.sqrt(n_old)), int(math.sqrt(n_new))
    if gs_old * gs_old != n_old or gs_new * gs_new != n_new:
        print(f'Skipping pos_embed resize: old={pos_embed_ckpt.shape}, new={pos_embed_model.shape}')
        return None
    pe = pos_embed_ckpt.reshape(1, gs_old, gs_old, c).permute(0, 3, 1, 2)
    pe = F.interpolate(pe, size=(gs_new, gs_new), mode='bicubic', align_corners=False)
    return pe.permute(0, 2, 3, 1).reshape(1, gs_new * gs_new, c)


def build_model_from_config(config_path, ckpt_path, data, device, img_size=None):
    """Build + load a Lightning model from a YAML config and a checkpoint."""
    with open(config_path, 'r') as f:
        cfg = yaml.safe_load(f)
    _img_size   = img_size if img_size is not None else data.img_size
    num_classes = data.num_classes
    encoder = _build_encoder(cfg, _img_size)
    network = _build_network(cfg, encoder, num_classes)
    model   = _build_lightning_module(cfg, network, _img_size, num_classes).eval().to(device)
    state_dict = torch.load(ckpt_path, map_location=device, weights_only=True)
    model.load_state_dict(state_dict, strict=False)
    return model, cfg


def build_coco_model(config_path, ckpt_path, device):
    """Build + load the COCO-panoptic EoMT model with filtered checkpoint."""
    with open(config_path, 'r') as f:
        cfg = yaml.safe_load(f)
    stuff_classes    = cfg['data']['init_args'].get('stuff_classes', [])
    coco_num_classes = 133 if len(stuff_classes) == 53 else (max(stuff_classes) + 1 if stuff_classes else 133)
    print(f'COCO inferred num_classes: {coco_num_classes}')
    encoder = _build_encoder(cfg, (640, 640))
    network = _build_network(cfg, encoder, coco_num_classes)
    model   = _build_lightning_module(cfg, network, (640, 640), coco_num_classes).eval().to(device)
    state_dict  = torch.load(ckpt_path, map_location=device, weights_only=True)
    model_state = model.state_dict()
    filtered, skipped = {}, []
    for k, v in state_dict.items():
        if k not in model_state:
            skipped.append((k, 'missing_in_model')); continue
        if v.shape == model_state[k].shape:
            filtered[k] = v; continue
        if k.endswith('network.encoder.backbone.pos_embed'):
            resized = _resize_abs_pos_embed(v, model_state[k])
            if resized is not None and resized.shape == model_state[k].shape:
                filtered[k] = resized
                print(f'Resized pos_embed: {tuple(v.shape)} -> {tuple(resized.shape)}')
                continue
        skipped.append((k, f'shape_mismatch {tuple(v.shape)} != {tuple(model_state[k].shape)}'))
    missing, unexpected = model.load_state_dict(filtered, strict=False)
    print(f'Loaded filtered checkpoint. Missing: {len(missing)} | Unexpected: {len(unexpected)} | Skipped: {len(skipped)}')
    return model, cfg, coco_num_classes


# ════════════════════════════════════════════════════════════
# 3b – Inference helpers
# ════════════════════════════════════════════════════════════

def infer_semantic(model_obj, img, model_img_size=None):
    """Unified semantic inference -> [1,1,H,W] prediction."""
    _img_size = model_img_size if model_img_size is not None else data.img_size
    with torch.no_grad(), autocast(dtype=torch.float16, device_type='cuda'):
        img_single = img.squeeze(0) if img.dim() == 4 else img
        imgs       = [img_single.to(device)]
        img_sizes  = [img_single.shape[-2:]]
        crops, origins = model_obj.window_imgs_semantic(imgs)
        mask_logits_layer, class_logits_layer = model_obj(crops)
        mask_logits = F.interpolate(
            mask_logits_layer[-1], _img_size, mode='bilinear', align_corners=False
        )
        crop_logits = model_obj.to_per_pixel_logits_semantic(mask_logits, class_logits_layer[-1])
        logits      = model_obj.revert_window_logits_semantic(crop_logits, origins, img_sizes)
        preds       = logits[0].argmax(0).cpu()
    return preds.unsqueeze(0).unsqueeze(0)  # [1,1,H,W]


# ════════════════════════════════════════════════════════════
# 3c – IoU evaluation helpers
# ════════════════════════════════════════════════════════════

from iouEval import iouEval, getColorEntry

def build_gt_from_batch(batch, ignore_label=IGNORE_LABEL):
    data_dict = batch[1][0]
    masks, train_ids = data_dict['masks'], data_dict['labels']
    H, W = masks.shape[-2], masks.shape[-1]
    gt = torch.full((1, H, W), ignore_label, dtype=torch.long)
    for i in range(len(masks)):
        t_id = int(train_ids[i])
        if 0 <= t_id <= 18:
            gt[0, masks[i]] = t_id
    return gt.unsqueeze(0)  # [1,1,H,W]


def remap_mask(preds, id_map, ignore_label=IGNORE_LABEL):
    remapped = torch.full_like(preds, ignore_label)
    for src_id, dst_id in id_map.items():
        remapped[preds == int(src_id)] = int(dst_id)
    return remapped


def evaluate_model(model_obj, model_name, prediction_adapter=None,
                   print_every=100, model_img_size=None):
    iou_eval = iouEval(data.num_classes + 1, ignoreIndex=IGNORE_LABEL)
    start = time.time()
    print(f'\nValidation - {model_name}')
    for step, batch in enumerate(val_loader, start=1):
        images    = batch[0][0]
        gt_labels = build_gt_from_batch(batch)
        pred      = infer_semantic(model_obj, images, model_img_size)
        if prediction_adapter is not None:
            pred = prediction_adapter(pred)
        iou_eval.addBatch(pred, gt_labels)
        if step % print_every == 0:
            print(f'  Step: {step}')
    iou_val, iou_classes = iou_eval.getIoU()
    print(f'  Took {time.time() - start:.1f}s')
    print(f'  Per-Class IoU - {model_name}:')
    for i, cls_name in enumerate(CITYSCAPES_CLASS_NAMES):
        print(f'    {cls_name:15s}: {float(iou_classes[i] * 100):6.2f}')
    print(f'  MEAN IoU - {model_name}: {float(iou_val * 100):.2f}%')
    return {'name': model_name, 'miou': float(iou_val), 'per_class_iou': iou_classes.cpu().numpy()}


# ════════════════════════════════════════════════════════════
# 3d – Anomaly Dataset (ERFNet: normalized; EoMT: raw uint8)
# ════════════════════════════════════════════════════════════

class AnomalyDataset(torch.utils.data.Dataset):
    """Used by ERFNet (Step 7): returns normalized [3,H,W] float tensor."""
    IMG_MEAN = [0.485, 0.456, 0.406]
    IMG_STD  = [0.229, 0.224, 0.225]
    IMG_EXTS = ('.png', '.jpg', '.jpeg', '.webp')

    def __init__(self, images_dir, masks_dir, mask_suffix='.png'):
        self.image_paths = sorted([os.path.join(images_dir, f)
                                    for f in os.listdir(images_dir)
                                    if f.lower().endswith(self.IMG_EXTS)])
        self.masks_dir, self.mask_suffix = masks_dir, mask_suffix
        self.transform = T.Compose([T.ToTensor(), T.Normalize(mean=self.IMG_MEAN, std=self.IMG_STD)])

    def __len__(self):  return len(self.image_paths)

    def __getitem__(self, idx):
        img_path  = self.image_paths[idx]
        stem      = os.path.splitext(os.path.basename(img_path))[0]
        mask_path = os.path.join(self.masks_dir, stem + self.mask_suffix)
        if not os.path.exists(mask_path):
            raise FileNotFoundError(f'Mask non trovata: {mask_path}')
        image = Image.open(img_path).convert('RGB')
        mask  = np.array(Image.open(mask_path))
        return self.transform(image), torch.from_numpy(mask).long(), stem


class EOMTAnomalyDataset(torch.utils.data.Dataset):
    IMG_EXTS = ('.png', '.jpg', '.jpeg', '.webp')

    def __init__(self, images_dir, masks_dir, mask_suffix='.png'):
        self.image_paths = sorted([os.path.join(images_dir, f)
                                    for f in os.listdir(images_dir)
                                    if f.lower().endswith(self.IMG_EXTS)])
        self.masks_dir, self.mask_suffix = masks_dir, mask_suffix

    def __len__(self):  return len(self.image_paths)

    def __getitem__(self, idx):
        img_path  = self.image_paths[idx]
        stem      = os.path.splitext(os.path.basename(img_path))[0]
        mask_path = os.path.join(self.masks_dir, stem + self.mask_suffix)
        if not os.path.exists(mask_path):
            raise FileNotFoundError(f'Mask non trovata: {mask_path}')
        image = np.array(Image.open(img_path).convert('RGB'), dtype=np.uint8)
        img_t = torch.from_numpy(image).permute(2, 0, 1).float()
        mask  = np.array(Image.open(mask_path))
        return img_t, torch.from_numpy(mask).long(), stem


# ════════════════════════════════════════════════════════════
# 3e – Anomaly score functions & metrics
# ════════════════════════════════════════════════════════════

def compute_metrics(scores: np.ndarray, labels: np.ndarray):
    """Return (AuPRC, FPR@95TPR) for flat binary score/label arrays."""
    auprc = average_precision_score(labels, scores)
    fpr, tpr, _ = roc_curve(labels, scores)
    idx   = np.searchsorted(tpr, 0.95)
    fpr95 = float(fpr[min(idx, len(fpr) - 1)])
    return round(auprc, 4), round(fpr95, 4)


def score_msp(logits, temperature=1.0):
    probs = F.softmax(logits / temperature, dim=1)
    return 1.0 - probs.max(dim=1).values.squeeze(0)


def score_max_logit(logits, temperature=1.0):
    return -(logits / temperature).max(dim=1).values.squeeze(0)


def score_max_entropy(logits, temperature=1.0):
    probs   = F.softmax(logits / temperature, dim=1)
    log_p   = F.log_softmax(logits / temperature, dim=1)
    entropy = -(probs * log_p).sum(dim=1)
    return (entropy / np.log(logits.shape[1])).squeeze(0)


def eomt_raw_to_sem_logits(mask_logits_raw, class_logits_raw, target_hw):
    """EoMT mask+class logits -> per-pixel semantic logits [1,C,H,W]."""
    ml = mask_logits_raw.unsqueeze(0)  if mask_logits_raw.dim()  == 3 else mask_logits_raw
    cl = class_logits_raw.unsqueeze(0) if class_logits_raw.dim() == 2 else class_logits_raw
    class_scores = F.softmax(cl[..., :-1], dim=-1)
    mask_sup     = F.interpolate(ml, size=target_hw, mode='bilinear', align_corners=False)
    mask_prob    = mask_sup.sigmoid()
    return torch.einsum('bqhw,bqc->bchw', mask_prob, class_scores)


def score_rba(mask_logits_raw, class_logits_raw, target_hw):
    """RbA: Rejected-by-All (Nayal et al., ICCV 2023)."""
    ml = mask_logits_raw.unsqueeze(0)  if mask_logits_raw.dim()  == 3 else mask_logits_raw
    cl = class_logits_raw.unsqueeze(0) if class_logits_raw.dim() == 2 else class_logits_raw
    class_scores = F.softmax(cl[..., :-1], dim=-1)
    max_cls      = class_scores.max(dim=-1).values  # [1,Q]
    mask_sup     = F.interpolate(ml, size=target_hw, mode='bilinear', align_corners=False)
    mask_prob    = mask_sup.sigmoid()
    support      = (mask_prob.clamp(min=0) * max_cls.unsqueeze(-1).unsqueeze(-1).clamp(min=0)).sum(dim=1)
    return -torch.log(support + 1e-6).squeeze(0)


# ════════════════════════════════════════════════════════════
# 3f – Dataset configs
# ════════════════════════════════════════════════════════════

DATASET_CONFIGS = {
    'SMIYC_RA21':  {'images': f'{ANOMALY_ROOT}/RoadAnomaly21/images',    'masks': f'{ANOMALY_ROOT}/RoadAnomaly21/labels_masks',    'mask_suffix': '.png'},
    'SMIYC_RO21':  {'images': f'{ANOMALY_ROOT}/RoadObstacle21/images',   'masks': f'{ANOMALY_ROOT}/RoadObstacle21/labels_masks',   'mask_suffix': '.png'},
    'FS_Static':   {'images': f'{ANOMALY_ROOT}/fs_static/images',        'masks': f'{ANOMALY_ROOT}/fs_static/labels_masks',        'mask_suffix': '.png'},
    'RoadAnomaly': {'images': f'{ANOMALY_ROOT}/RoadAnomaly/images',      'masks': f'{ANOMALY_ROOT}/RoadAnomaly/labels_masks',      'mask_suffix': '.png'},
    'FS_LnF':      {'images': f'{ANOMALY_ROOT}/FS_LostFound_full/images','masks': f'{ANOMALY_ROOT}/FS_LostFound_full/labels_masks','mask_suffix': '.png'},
}

for ds_name, cfg_ds in DATASET_CONFIGS.items():
    n = len(os.listdir(cfg_ds['images'])) if os.path.isdir(cfg_ds['images']) else 'PATH NON TROVATO'
    print(f'  {ds_name:<15}: {n} immagini')

print('\nUtility functions OK.')


# Step 4 – Load Models

In [ ]:
model, config_city = build_model_from_config(CONFIG_CITY, CKPT_CITY, data, device)
print('Cityscapes model loaded.')

model_coco, config_coco, coco_num_classes = build_coco_model(CONFIG_COCO, CKPT_COCO, device)
print('COCO model loaded correctly.')


## Step 4b – COCO → Cityscapes Mapping & Evaluation

In [ ]:
COCO_TO_CITYSCAPES = {
    # THINGS
    0: 11,   # person
    1: 18,   # bicycle
    2: 13,   # car
    3: 17,   # motorcycle
    5: 15,   # bus
    6: 16,   # train
    7: 14,   # truck
    9:  6,   # traffic light
    11:  7,  # stop sign -> traffic sign
    # STUFF
    91:  2,  # house -> building
    92:  5,  # light -> pole
    98: 10,  # railroad -> terrain
    100:  0, # road
    102:  9, # sand -> terrain
    109:  3, # wall-brick
    110:  3, # wall-stone
    111:  3, # wall-tile
    112:  3, # wall-wood
    116:  8, # tree-merged -> vegetation
    117:  4, # fence-merged
    119: 10, # sky-other-merged -> sky
    123:  1, # pavement-merged -> sidewalk
    124:  9, # mountain-merged -> terrain
    125:  8, # grass-merged -> vegetation
    126:  9, # dirt-merged -> terrain
    129:  2, # building-other-merged -> building
    130:  9, # rock-merged -> terrain
    131:  3, # wall-other-merged -> wall
}

def coco_to_cityscapes_adapter(pred):
    return remap_mask(pred, COCO_TO_CITYSCAPES)


results_city = evaluate_model(model, 'Cityscapes model', print_every=100)

if isinstance(model_coco.img_size, int):
    model_coco.img_size = (model_coco.img_size, model_coco.img_size)

results_coco = evaluate_model(
    model_coco, 'COCO panoptic model mapped to Cityscapes',
    model_img_size=(640, 640), prediction_adapter=coco_to_cityscapes_adapter, print_every=100,
)

print('\n================ FINAL COMPARISON ================')
print(f"Cityscapes model mIoU : {results_city['miou'] * 100:.2f}")
print(f"COCO model mIoU       : {results_coco['miou'] * 100:.2f}")


# Step 5 – Fine-tuning: COCO → Cityscapes (Phase 1 – head only)

In [ ]:
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor, Callback
from lightning.pytorch.loggers import WandbLogger

# Datamodule
with open(CONFIG_CITY, 'r') as f:
    cfg_city = yaml.safe_load(f)

dm_name, dm_cls_name = cfg_city['data']['class_path'].rsplit('.', 1)
dm_cls    = getattr(importlib.import_module(dm_name), dm_cls_name)
dm_kwargs = cfg_city['data'].get('init_args', {})

data_ft = dm_cls(
    path=DATA_PATH, batch_size=2, num_workers=2, check_empty_targets=False,
    img_size=(640, 640), **{k: v for k, v in dm_kwargs.items() if k != 'img_size'},
)
data_ft.setup()
print(f'Cityscapes train datamodule ready. num_classes={data_ft.num_classes} img_size={data_ft.img_size}')

# Build model: Cityscapes head + COCO backbone
encoder_ft    = _build_encoder(cfg_city, (640, 640))
network_ft    = _build_network(cfg_city, encoder_ft, data_ft.num_classes)
lit_mod, lit_cls_name = cfg_city['model']['class_path'].rsplit('.', 1)
lit_cls       = getattr(importlib.import_module(lit_mod), lit_cls_name)
model_kwargs_ft = {k: v for k, v in cfg_city['model']['init_args'].items() if k != 'network'}
if 'stuff_classes' in cfg_city['data'].get('init_args', {}):
    model_kwargs_ft['stuff_classes'] = cfg_city['data']['init_args']['stuff_classes']
model_ft = lit_cls(img_size=(640, 640), num_classes=data_ft.num_classes, network=network_ft, **model_kwargs_ft)

# Load COCO backbone weights
coco_state  = torch.load(CKPT_COCO, map_location='cpu', weights_only=True)
model_state = model_ft.state_dict()
filtered, skipped = {}, []
for k, v in coco_state.items():
    if k not in model_state: skipped.append((k, 'missing')); continue
    if tuple(v.shape) == tuple(model_state[k].shape): filtered[k] = v
    else: skipped.append((k, f'shape {tuple(v.shape)} != {tuple(model_state[k].shape)}'))
missing, unexpected = model_ft.load_state_dict(filtered, strict=False)
print(f'Transferred: {len(filtered)} tensors | Missing: {len(missing)} | Skipped: {len(skipped)}')

# Phase 1: freeze backbone, train head only
HEAD_KEYWORDS = ['class_head', 'mask_head', 'upscale', 'q.weight']

def set_trainable(model, head_keywords):
    frozen, trainable = 0, 0
    for name, param in model.named_parameters():
        param.requires_grad = any(kw in name for kw in head_keywords)
        if param.requires_grad: trainable += 1
        else:                   frozen    += 1
    print(f'Trainable params: {trainable} | Frozen params: {frozen}')

set_trainable(model_ft, HEAD_KEYWORDS)
print('\nTrainable layers (sample):')
for name, p in model_ft.named_parameters():
    if p.requires_grad: print(f'  {name}')


In [ ]:
from google.colab import userdata
import wandb
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
wandb.login()


In [ ]:
# Ensure hparams consistency
if hasattr(model_ft, 'lr'):            model_ft.lr            = 1e-4
if hasattr(model_ft, 'learning_rate'): model_ft.learning_rate = 1e-4
if hasattr(model_ft, 'hparams'):       model_ft.hparams['img_size'] = (640, 640)
if hasattr(data_ft,  'hparams'):       data_ft.hparams['img_size']  = (640, 640)

class TimingCallback(Callback):
    def on_fit_start(self, trainer, pl_module):
        self.t0 = time.time(); print('TRAIN Fit started')
    def on_train_epoch_start(self, trainer, pl_module):
        self.epoch_t0 = time.time()
        print(f'TRAIN Epoch {trainer.current_epoch + 1}/{trainer.max_epochs} started')
    def on_train_epoch_end(self, trainer, pl_module):
        dt = time.time() - self.epoch_t0
        print(f'TRAIN Epoch {trainer.current_epoch + 1} ended in {dt/60:.2f} min')

checkpoint_cb = ModelCheckpoint(
    dirpath=SAVE_DIR,
    filename='phase1-epoch{epoch:02d}-val_miou{metrics/val_iou_all:.4f}',
    save_top_k=2, monitor='metrics/val_iou_all', mode='max', save_last=True,
)
lr_monitor   = LearningRateMonitor(logging_interval='epoch')
wandb_logger = WandbLogger(project='faimdl', name='phase1-head-only', save_dir=SAVE_DIR)

trainer = L.Trainer(
    accelerator='gpu' if torch.cuda.is_available() else 'cpu', devices=1,
    max_epochs=15, precision='16-mixed' if torch.cuda.is_available() else '32-true',
    logger=wandb_logger, callbacks=[checkpoint_cb, lr_monitor, TimingCallback()],
    log_every_n_steps=20, val_check_interval=1.0, gradient_clip_val=1.0,
)

last_ckpt = os.path.join(SAVE_DIR, 'last.ckpt')
print('Starting Phase 1 fine-tuning...')
trainer.fit(model_ft, datamodule=data_ft, ckpt_path=last_ckpt if os.path.exists(last_ckpt) else None)


# Step 6 – Evaluate Fine-tuned Model

In [ ]:
best_ckpt = checkpoint_cb.best_model_path
print(f'Loading best checkpoint: {best_ckpt}')

encoder_eval  = _build_encoder(cfg_city, (640, 640))
network_eval  = _build_network(cfg_city, encoder_eval, data_ft.num_classes)
model_ft_eval = lit_cls(
    img_size=(640, 640), num_classes=data_ft.num_classes,
    network=network_eval, **model_kwargs_ft,
).eval().to(device)

ckpt_data = torch.load(best_ckpt, map_location=device, weights_only=False)
model_ft_eval.load_state_dict(ckpt_data.get('state_dict', ckpt_data), strict=False)

results_ft = evaluate_model(
    model_ft_eval, 'Fine-tuned COCO->Cityscapes (head only)',
    model_img_size=(640, 640), print_every=100,
)

print('\n================ STEP 5 COMPARISON ================')
print(f"Cityscapes baseline  : {results_city['miou'] * 100:.2f}")
print(f"COCO zero-shot mapped: {results_coco['miou'] * 100:.2f}")
print(f"COCO fine-tuned      : {results_ft['miou']   * 100:.2f}")


# Step 7 – Anomaly Detection: ERFNet Baseline


In [ ]:
from erfnet import ERFNet

erfnet    = ERFNet(num_classes=20)
state_raw = torch.load(ERFNET_WEIGHTS, map_location='cpu', weights_only=True)
state_clean = {k.replace('module.', ''): v for k, v in state_raw.items()}
missing, unexpected = erfnet.load_state_dict(state_clean, strict=False)
print(f'Missing keys ({len(missing)}): {missing}')
print(f'Unexpected keys ({len(unexpected)}): {unexpected}')
erfnet = erfnet.eval().to(device)
print('ERFNet loaded')


In [ ]:
# ── Constants & chunk helpers ─────────────────────────────────────────────
CHUNK_SIZE    = 10
METHODS_STEP7 = {'MSP': score_msp, 'MaxLogit': score_max_logit, 'MaxEntropy': score_max_entropy}

def _s7_cache_dir(ds_name):
    d = Path(STEP7_CACHE_DIR) / ds_name; d.mkdir(parents=True, exist_ok=True); return d

def _s7_chunk_path(ds_name, i): return _s7_cache_dir(ds_name) / f'chunk_{i:04d}.npz'
def _s7_meta_path(ds_name):     return _s7_cache_dir(ds_name) / 'meta.json'


def process_dataset_to_chunks_s7(ds_name, cfg):
    dataset = AnomalyDataset(cfg['images'], cfg['masks'], cfg['mask_suffix'])
    loader  = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)
    meta    = {'num_samples': len(dataset), 'num_chunks': 0, 'chunk_size': CHUNK_SIZE}
    with open(_s7_meta_path(ds_name), 'w') as f: json.dump(meta, f, indent=2)

    chunk_scores = {m: [] for m in METHODS_STEP7}
    chunk_labels, chunk_idx, sample_in_chunk = [], 0, 0

    def _flush():
        nonlocal chunk_idx, sample_in_chunk
        arrays = {m: np.concatenate(chunk_scores[m]) if chunk_scores[m] else np.array([], dtype=np.float32)
                  for m in METHODS_STEP7}
        labels = np.concatenate(chunk_labels) if chunk_labels else np.array([], dtype=np.uint8)
        np.savez_compressed(_s7_chunk_path(ds_name, chunk_idx), **arrays, labels=labels)
        chunk_idx += 1
        for m in METHODS_STEP7: chunk_scores[m].clear()
        chunk_labels.clear(); sample_in_chunk = 0
        gc.collect(); torch.cuda.empty_cache()

    erfnet.eval()
    with torch.no_grad():
        for img, mask, _ in tqdm(loader, desc=f'ERFNet [{ds_name}]'):
            img = img.to(device)
            with autocast(dtype=torch.float16, device_type='cuda'):
                logits = erfnet(img)[:, :19]
            if logits.shape[-2:] != mask.shape[-2:]:
                logits = F.interpolate(logits, size=mask.shape[-2:], mode='bilinear', align_corners=False)
            logits_cpu = logits.float().cpu()
            mask_np    = mask.squeeze(0).numpy()
            valid      = mask_np != 255
            for m_name, fn in METHODS_STEP7.items():
                chunk_scores[m_name].append(fn(logits_cpu).numpy()[valid].astype(np.float32, copy=False))
            chunk_labels.append(mask_np[valid].astype(np.uint8, copy=False))
            sample_in_chunk += 1
            if sample_in_chunk >= CHUNK_SIZE: _flush()
    if sample_in_chunk > 0: _flush()
    meta['num_chunks'] = chunk_idx
    with open(_s7_meta_path(ds_name), 'w') as f: json.dump(meta, f, indent=2)


def evaluate_dataset_from_chunks_s7(ds_name):
    meta_file = _s7_meta_path(ds_name)
    if not meta_file.exists(): raise FileNotFoundError(f'Missing meta for {ds_name}: {meta_file}')
    with open(meta_file) as f: meta = json.load(f)
    all_scores = {m: [] for m in METHODS_STEP7}; all_labels = []
    for i in range(meta['num_chunks']):
        d = np.load(_s7_chunk_path(ds_name, i))
        for m in METHODS_STEP7:
            if d[m].size: all_scores[m].append(d[m])
        if d['labels'].size: all_labels.append(d['labels'])
    labels_flat = np.concatenate(all_labels) if all_labels else np.array([], dtype=np.uint8)
    out = {}
    for m in METHODS_STEP7:
        scores_flat = np.concatenate(all_scores[m]) if all_scores[m] else np.array([], dtype=np.float32)
        auprc, fpr95 = compute_metrics(scores_flat, labels_flat)
        out[m] = {'AuPRC': float(auprc), 'FPR95': float(fpr95)}
    return out


# ── Main loop ────────────────────────────────────────────────────────────
results_step7 = json.load(open(STEP7_RESULTS)) if os.path.exists(STEP7_RESULTS) else {}

for ds_name, cfg_ds in DATASET_CONFIGS.items():
    if ds_name in results_step7:
        print(f'[SKIP] {ds_name} gia presente in {STEP7_RESULTS}'); continue
    print(f'\n{"="*55}\nDataset: {ds_name}')
    if not _s7_meta_path(ds_name).exists():
        process_dataset_to_chunks_s7(ds_name, cfg_ds)
    else:
        print('  chunk gia presenti')
    results_step7[ds_name] = evaluate_dataset_from_chunks_s7(ds_name)
    with open(STEP7_RESULTS, 'w') as f: json.dump(results_step7, f, indent=2)
    print(f'  Salvato: {STEP7_RESULTS}')
    gc.collect(); torch.cuda.empty_cache()

print('\nSTEP 7 COMPLETATO')
print(json.dumps(results_step7, indent=2))


# Step 8 – Anomaly Detection: EoMT Models


In [ ]:
# ── Checkpoints to be evaluated ──────────────────────────────────────────────
EOMT_CHECKPOINTS = {
    'EoMT-Cityscapes': {'config': CONFIG_CITY, 'ckpt': CKPT_CITY},
    'EoMT-FineTuned':  {'config': CONFIG_CITY, 'ckpt': checkpoint_cb.best_model_path},
}

METHODS_STEP8   = ['MSP', 'MaxLogit', 'MaxEntropy', 'RbA']
MSP_TS          = 'MSP_TS'
TEMPERATURES    = [0.5, 1.0, 2.0, 5.0, 10.0]
STEP8_CHUNKSIZE = 5


# ── JSON helpers ─────────────────────────────────────────────────────────
def _step8_load_results():
    return json.load(open(STEP8_RESULTS)) if os.path.exists(STEP8_RESULTS) else {}

def _step8_save_result(ckpt_name, ds_name, method, value):
    res = _step8_load_results()
    res.setdefault(ckpt_name, {}).setdefault(ds_name, {})[method] = value
    with open(STEP8_RESULTS, 'w') as f: json.dump(res, f, indent=2)


# ── Cache EoMT logits per image ───────────────────────────────────────────
def stream_eomt_outputs_to_chunks(model_obj, ckpt_name, ds_name, cfg,
                                   cache_subdir, method_name, temperature=1.0, chunk_size=5):
    dataset = EOMTAnomalyDataset(cfg['images'], cfg['masks'], cfg['mask_suffix'])
    loader  = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)
    model_obj.eval()
    with torch.no_grad():
        for img, mask, stem in tqdm(loader, desc=f'cache {ds_name}'):
            stem    = stem[0]
            outpath = Path(cache_subdir) / f'{stem}.pt'
            if outpath.exists(): continue
            img_np = img.squeeze(0).detach().cpu().permute(1, 2, 0).numpy()
            img_np = np.clip(img_np, 0, 255).astype(np.uint8)
            img_t  = torch.from_numpy(img_np).permute(2, 0, 1).contiguous().to(torch.uint8)
            with autocast(dtype=torch.float16, device_type='cuda'):
                crops, origins = model_obj.window_imgs_semantic([img_t])
                crops = crops.to(device) if isinstance(crops, torch.Tensor) else [c.to(device) for c in crops]
                mask_logits_layer, class_logits_layer = model_obj(crops)
            torch.save({
                'mask_logits':  [x.float().cpu() for x in mask_logits_layer],
                'class_logits': [x.float().cpu() for x in class_logits_layer],
                'origins': origins, 'mask': mask.squeeze(0).cpu(),
                'stem': stem, 'orig_shape': tuple(mask.shape[-2:]),
            }, outpath)


# ── Evaluate one method from cached .pt files ─────────────────────────────
def evaluate_eomt_method_from_chunks(ckpt_name, ds_name, method_name, temperature=1.0):
    cache_subdir = Path(STEP8_CACHE_DIR) / ckpt_name / ds_name
    files        = sorted(cache_subdir.glob('*.pt'))
    if not files: raise FileNotFoundError(f'Nessun file .pt in {cache_subdir}')
    all_scores, all_labels = [], []
    fn_map = {'MSP': score_msp, MSP_TS: score_msp,
              'MaxLogit': score_max_logit, 'MaxEntropy': score_max_entropy}
    for f in tqdm(files, desc=f'eval {method_name} T={temperature}', leave=False):
        if f.name == 'meta.json': continue
        item = torch.load(f, map_location='cpu')
        ml, cl = item['mask_logits'][-1].to(device), item['class_logits'][-1].to(device)
        gt     = item['mask'].numpy().astype(np.int64)
        target_hw = tuple(gt.shape)
        if method_name == 'RbA':
            scores = score_rba(ml, cl, target_hw).cpu().numpy()
        else:
            sem_logits = eomt_raw_to_sem_logits(ml, cl, target_hw)
            scores     = fn_map[method_name](sem_logits, temperature).cpu().numpy()
        if scores.ndim == 3: scores = scores.max(axis=0)
        elif scores.ndim == 4: scores = scores.max(axis=1).squeeze(0)
        valid  = gt != 255
        labels = (gt > 0).astype(np.uint8)
        if np.any(valid):
            all_scores.append(scores[valid]); all_labels.append(labels[valid])
    if not all_scores: raise ValueError(f'Nessun pixel valido in {cache_subdir}')
    return compute_metrics(np.concatenate(all_scores), np.concatenate(all_labels))


print('Step 8 helpers OK.')


In [ ]:
# ════════════════════════════════════════════════════════════
# Step 8 – Main evaluation loop
# ════════════════════════════════════════════════════════════

results_step8 = _step8_load_results()

for ckpt_name, ckpt_cfg in EOMT_CHECKPOINTS.items():
    print(f'\n{"="*60}\n  Checkpoint: {ckpt_name}\n{"="*60}')
    model_eomt, _ = build_model_from_config(
        config_path=ckpt_cfg['config'], ckpt_path=ckpt_cfg['ckpt'], data=data, device=device,
    )
    model_eomt.eval()
    print('  Modello caricato.')

    for ds_name, cfg_ds in DATASET_CONFIGS.items():
        print(f'\n  Dataset: {ds_name}')
        cache_subdir       = os.path.join(STEP8_CACHE_DIR, ckpt_name, ds_name)
        os.makedirs(cache_subdir, exist_ok=True)
        current_chunk_size = 1 if ds_name == 'FS_LnF' else STEP8_CHUNKSIZE

        # Standard methods
        for method in METHODS_STEP8:
            if results_step8.get(ckpt_name, {}).get(ds_name, {}).get(method):
                r = results_step8[ckpt_name][ds_name][method]
                print(f'    SKIP {method} (AuPRC={r["AuPRC"]:.4f})')
                continue
            stream_eomt_outputs_to_chunks(
                model_eomt, ckpt_name, ds_name, cfg_ds, cache_subdir,
                method, temperature=1.0, chunk_size=current_chunk_size,
            )
            auprc, fpr95 = evaluate_eomt_method_from_chunks(ckpt_name, ds_name, method, 1.0)
            print(f'    {method:<15} AuPRC={auprc:.4f}  FPR95={fpr95:.4f}')
            _step8_save_result(ckpt_name, ds_name, method, {'AuPRC': float(auprc), 'FPR95': float(fpr95)})
            results_step8 = _step8_load_results()

        # Temperature Scaling on MSP
        print(f'\n  Temperature Scaling MSP ({ds_name})')
        best_t, best_auprc = 1.0, -1.0
        for t in TEMPERATURES:
            meta_key_ts = f'MSP_T{t}'
            if results_step8.get(ckpt_name, {}).get(ds_name, {}).get(meta_key_ts):
                existing = results_step8[ckpt_name][ds_name][meta_key_ts]
                marker   = '  best' if existing['AuPRC'] > best_auprc else ''
                print(f'    T={t:<5}  AuPRC={existing["AuPRC"]:.4f}  FPR95={existing["FPR95"]:.4f}  (gia presente){marker}')
                if existing['AuPRC'] > best_auprc: best_auprc, best_t = existing['AuPRC'], t
                continue
            stream_eomt_outputs_to_chunks(
                model_eomt, ckpt_name, ds_name, cfg_ds, cache_subdir,
                MSP_TS, temperature=t, chunk_size=current_chunk_size,
            )
            auprc, fpr95 = evaluate_eomt_method_from_chunks(ckpt_name, ds_name, MSP_TS, t)
            marker = '  best' if auprc > best_auprc else ''
            print(f'    T={t:<5}  AuPRC={auprc:.4f}  FPR95={fpr95:.4f}{marker}')
            _step8_save_result(ckpt_name, ds_name, meta_key_ts, {'AuPRC': float(auprc), 'FPR95': float(fpr95)})
            results_step8 = _step8_load_results()
            if auprc > best_auprc: best_auprc, best_t = auprc, t
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()

        _step8_save_result(ckpt_name, ds_name, 'best_T_MSP', {'best_T': best_t, 'best_AuPRC': best_auprc})
        results_step8 = _step8_load_results()
        print(f'    -> Best T per {ds_name}: T={best_t}  AuPRC={best_auprc:.4f}')

    del model_eomt; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print('\n' + '='*60 + '\n  Step 8 completato.\n' + '='*60)
print(json.dumps(results_step8, indent=2))
